In [ ]:
import os
import re
import json
import time
import random
from collections import Counter
from typing import List
import pandas as pd
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from colorama import Fore, Style, init

init(autoreset=True)

# ============================================================
# Logging utilities
# ============================================================
def log(msg: str):
    print(time.strftime("[%H:%M:%S]"), msg, flush=True)

def pretty_log(stage, text):
    """Colored structured logging"""
    print(f"{Fore.CYAN}[{stage}] {Style.RESET_ALL}{text}", flush=True)

SEPARATOR = "\n" + "*" * 55 + "\n"

# ============================================================
# Device preparation
# ============================================================
def prepare_device(use_cpu: bool = False):
    if use_cpu:
        return torch.device("cpu"), torch.float32
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available. Use use_cpu=True to run on CPU.")
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    return torch.device("cuda"), torch.float16

# ============================================================
# Model loading
# ============================================================
def load_model(model_path: str, device: torch.device, torch_dtype: torch.dtype):
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch_dtype)
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token
    model = model.to(device)
    model.eval()
    return tokenizer, model

# ============================================================
# Text generation
# ============================================================
def generate_text(tokenizer, model, device, prompt: str,
                  temperature: float = 0.7, max_new_tokens: int = 900) -> str:
    try:
        inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=True)
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs.get("attention_mask", torch.ones_like(input_ids)).to(device)
        input_len = input_ids.shape[-1]

        with torch.no_grad():
            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        generated = outputs[0][input_len:]
        text = tokenizer.decode(generated, skip_special_tokens=True)
        return text.strip()
    except Exception as e:
        log(f"Generation error: {e}")
        return ""

# ============================================================
# LaTeX → readable math-text converter
# ============================================================
def latex_to_mathtext(eq: str) -> str:
    if not eq or not isinstance(eq, str):
        return eq
    s = eq
    s = s.replace("\\left", "").replace("\\right", "")
    s = s.replace("\\cdot", "*").replace("\\,", "")
    s = re.sub(r"\\frac\{([^{}]+)\}\{([^{}]+)\}", r"(\1)/(\2)", s)
    s = re.sub(r"\\sqrt\{([^{}]+)\}", r"√(\1)", s)
    s = re.sub(r"\\sin\{?([^{}]+)\}?", r"sin(\1)", s)
    s = re.sub(r"\\cos\{?([^{}]+)\}?", r"cos(\1)", s)
    s = re.sub(r"\\tan\{?([^{}]+)\}?", r"tan(\1)", s)
    s = re.sub(r"\\arccos\{?([^{}]+)\}?", r"arccos(\1)", s)
    s = re.sub(r"\\arcsin\{?([^{}]+)\}?", r"arcsin(\1)", s)
    s = re.sub(r"\\arctan\{?([^{}]+)\}?", r"arctan(\1)", s)
    s = re.sub(r"\\exp\{?([^{}]+)\}?", r"exp(\1)", s)
    s = re.sub(r"\\log\{?([^{}]+)\}?", r"log(\1)", s)
    s = re.sub(r"\\operatorname\{?acos\}?", "arccos", s)
    s = re.sub(r"\\operatorname\{?asin\}?", "arcsin", s)
    s = re.sub(r"\\operatorname\{?atan\}?", "arctan", s)
    s = s.replace("\\prime", "'")
    s = re.sub(r"\s+", " ", s)
    s = s.replace("{", "").replace("}", "").replace("\\", "")
    s = s.strip()
    return s

# ============================================================
# LaTeX normalization for equation parsing
# ============================================================
def normalize_latex_equation(eq: str) -> str:
    if not eq:
        return eq
    eq = eq.strip().replace("\\left", "").replace("\\right", "")
    eq = re.sub(r"y\^\{\\prime\\prime\}", "y''", eq)
    eq = re.sub(r"y\^\{\\prime\}", "y'", eq)
    eq = re.sub(r"\\frac\{([^{}]+)\}\{([^{}]+)\}", r"(\1)/(\2)", eq)
    eq = re.sub(r"\\cos\s*(?:\{([^}]*)\}|\(([^)]*)\))", lambda m: f"cos({m.group(1) or m.group(2)})", eq)
    eq = re.sub(r"\\sin\s*(?:\{([^}]*)\}|\(([^)]*)\))", lambda m: f"sin({m.group(1) or m.group(2)})", eq)
    eq = re.sub(r"\\tan\s*(?:\{([^}]*)\}|\(([^)]*)\))", lambda m: f"tan({m.group(1) or m.group(2)})", eq)
    eq = re.sub(r"[{}]", "", eq)
    eq = eq.replace("\\", "")
    return eq.strip()

# ============================================================
# Prompts
# ============================================================
def build_seed_prompt(eq: str) -> str:
    return (
        "You are given a differential equation:\n"
        f"{eq}\n\n"
        "Find the general symbolic solution y(x). "
        "Finish strictly with \\boxed{y=...+C}. Do not add any text after the box.\n"
        "Solution:\n"
    )

def build_refine_prompt(equation: str, y_h: str, y_p_guess: str) -> str:
    return f"""
You are a symbolic ODE solver.

Original equation:
{equation}

Homogeneous part (fixed):
y_h(x) = {y_h if y_h else "0"}

Particular part to verify:
y_p(x) = {y_p_guess if y_p_guess else "0"}

Steps:
1. Form y(x) = y_h + y_p.
2. Compute y'(x), y''(x).
3. Substitute into ODE and simplify.
4. If mismatch, correct numeric coefficients in y_p(x) only.
5. Output the corrected full general solution as \\boxed{{y=...}} (nothing else).
"""

# ============================================================
# Token extraction
# ============================================================
def extract_final_token(text: str) -> str:
    if not text:
        return ""
    start = text.rfind("\\boxed{")
    if start == -1:
        return ""
    i = start + len("\\boxed{")
    depth, result_chars = 1, []
    while i < len(text) and depth > 0:
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                break
        if depth > 0:
            result_chars.append(ch)
        i += 1
    result = "".join(result_chars).strip()
    if not result.startswith("y"):
        result = "y = " + result
    return result

def normalize_for_vote(ans: str) -> str:
    s = ans.strip()
    s = re.sub(r"\s+", " ", s)
    return s

# ============================================================
# Homogeneous / particular separation
# ============================================================
def split_hom_particular(token: str):
    s = token.strip()
    s = re.sub(r'^\s*y\s*=\s*', '', s)
    terms = re.findall(r'[+\-]?\s*[^+\-]+', s)
    hom_terms, part_terms = [], []
    for t in terms:
        if re.search(r'C\s*_?\s*\d', t):
            hom_terms.append(t.strip())
        else:
            part_terms.append(t.strip())
    y_h = " + ".join(hom_terms).strip()
    y_p = " + ".join(part_terms).strip()
    return y_h, y_p

# ============================================================
# RSA reasoning loop
# ============================================================
def rsa_reasoning(tokenizer, model, device, equation: str,
                  N=3, K=2, T=2, temperature=0.7, max_new_tokens=1000, seed=42):
    rng = random.Random(seed)
    log(f"{SEPARATOR}\n[🧩] Start solving ODE → {equation}\n{SEPARATOR}")

    # Step 0: Initial population
    pop = []
    seed_prompt = build_seed_prompt(equation)
    for i in range(N):
        pretty_log("GEN", f"Generating candidate {i+1}/{N} ...")
        resp = generate_text(tokenizer, model, device, seed_prompt, temperature, max_new_tokens)
        pop.append(resp)
        preview = extract_final_token(resp)
        if preview:
            pretty_log("CAND", f"{Fore.YELLOW}{latex_to_mathtext(preview)}{Style.RESET_ALL}")

    # Aggregation iterations
    for t in range(T):
        new_pop = []
        for i in range(N):
            chosen = rng.sample(pop, k=min(K, len(pop)))
            agg_prompt = "You are given multiple candidate solutions to one ODE. Combine them into a single correct one.\n"
            agg_prompt += f"Equation:\n{equation}\n\n"
            for j, cand in enumerate(chosen, 1):
                agg_prompt += f"Candidate #{j}:\n{cand}\n\n"
            agg_prompt += "Output one improved solution ending with \\boxed{y=...+C}.\n"
            agg_resp = generate_text(tokenizer, model, device, agg_prompt, temperature, max_new_tokens)
            new_pop.append(agg_resp)
        pop = new_pop

    # Majority vote
    tokens = [normalize_for_vote(extract_final_token(p)) for p in pop if p]
    valid_tokens = [t for t in tokens if t]
    if not valid_tokens:
        pretty_log("WARN", "⚠️ No valid \\boxed{} results.")
        return "", pop, tokens
    final_token, count = Counter(valid_tokens).most_common(1)[0]
    pretty_log("RESULT", f"🏁 Majority: {Fore.GREEN}{latex_to_mathtext(final_token)}{Style.RESET_ALL}")

    # Refinement
    y_h, y_p_guess = split_hom_particular(final_token)
    refine_prompt = build_refine_prompt(equation, y_h, y_p_guess)
    refined_resp = generate_text(tokenizer, model, device, refine_prompt, 0.4, 800)
    refined_token = extract_final_token(refined_resp)
    if refined_token:
        refined_token = normalize_for_vote(refined_token)
        pretty_log("REFINED", f"✅ Refined: {Fore.GREEN}{latex_to_mathtext(refined_token)}{Style.RESET_ALL}")
        final_token = refined_token

    pretty_log("FINAL", f"📦 {Fore.CYAN}{latex_to_mathtext(final_token)}{Style.RESET_ALL}")
    return final_token, pop, tokens

# ============================================================
# Pipeline
# ============================================================
def run_pipeline(input_path: str, output_path: str, model_path: str,
                 N=3, K=2, T=2, temperature=0.7, max_new_tokens=1200,
                 data_fraction=1.0, save_json=None, use_cpu=False, seed=42):

    device, torch_dtype = prepare_device(use_cpu)
    log(f"Device: {device}, dtype: {torch_dtype}")
    tokenizer, model = load_model(model_path, device, torch_dtype)
    log(f"Model loaded: {model_path}")

    # Excel loading (auto header detection)
    try:
        df = pd.read_excel(input_path, header=0)
        if "equation" not in df.columns:
            df = pd.read_excel(input_path, header=None, names=["equation", "true_answer", "type_eq"])
    except Exception as e:
        log(f"⚠️ Excel read error: {e}")
        df = pd.read_excel(input_path, header=None, names=["equation", "true_answer", "type_eq"])

    df = df.dropna(subset=["equation"]).reset_index(drop=True)
    pretty_log("INFO", f"📄 Loaded {len(df)} equations.")

    results, gen_col = [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="RSA-solving"):
        eq_raw = str(row.get("equation", "")).strip()
        eq_norm = normalize_latex_equation(eq_raw)
        true_ans = str(row.get("true_answer", "")).strip()
        type_eq = str(row.get("type_eq", ""))
        if not eq_raw:
            continue
        final_token, _, _ = rsa_reasoning(tokenizer, model, device, eq_norm,
                                          N, K, T, temperature, max_new_tokens, seed)
        gen_col.append(final_token)
        results.append({
            "equation": eq_raw,
            "equation_math": latex_to_mathtext(eq_raw),
            "equation_normalized": eq_norm,
            "true_answer": true_ans,
            "true_answer_math": latex_to_mathtext(true_ans),
            "final_answer": final_token,
            "final_answer_math": latex_to_mathtext(final_token),
            "equation_type": type_eq
        })

    df["generated_answer"] = gen_col
    df["generated_answer_math"] = df["generated_answer"].apply(latex_to_mathtext)

    # Save outputs
    os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
    df.to_excel(output_path, index=False)
    log(f"✅ XLSX saved: {output_path}")

    if save_json:
        with open(save_json, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        log(f"📝 JSON saved: {save_json}")

# ============================================================
# Run
# ============================================================
INPUT_XLSX  = r"D:\koltcov\lab_related_2025\РНФ_2023-25\improve_models\test_x_small.xlsx"
OUTPUT_XLSX = r"D:\koltcov\lab_related_2025\РНФ_2023-25\improve_models\out_RSA.xlsx"
MODEL_PATH  = r"D:\koltcov\lab_related_2025\РНФ_2023-25\2_year\Open-Reasoner-Zero-1.5B\Open-Reasoner-Zero-1.5B_model"
SAVE_JSON   = r"D:\koltcov\lab_related_2025\РНФ_2023-25\improve_models\results_RSA_majority.json"

PARAMS = dict(N=3, K=2, T=2, temperature=0.7, max_new_tokens=2000,
              data_fraction=1.0, save_json=SAVE_JSON, use_cpu=False, seed=42)

random.seed(PARAMS["seed"])
torch.manual_seed(PARAMS["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(PARAMS["seed"])

run_pipeline(INPUT_XLSX, OUTPUT_XLSX, MODEL_PATH, **PARAMS)


[17:24:06] Device: cuda, dtype: torch.float16
[17:24:11] Model loaded: D:\koltcov\lab_related_2025\РНФ_2023-25\2_year\Open-Reasoner-Zero-1.5B\Open-Reasoner-Zero-1.5B_model
[INFO] 📄 Loaded 4 equations.


RSA-solving:   0%|                                                                               | 0/4 [00:00<?, ?it/s]

[17:24:12] 
*******************************************************

[🧩] Start solving ODE → -3y'' -y' -3y = cos(x)

*******************************************************

[GEN] Generating candidate 1/3 ...
[CAND] y = e^-(x)/(6) ( C_1 cos((frac)√(35)6 x) + C_2 sin((frac)√(35)6 x) ) + (3)/(8) cos((x) + (1)/(8) sin((x)))
[GEN] Generating candidate 2/3 ...
[CAND] y(x) = e^-(x)/(6) ( C_1 cos((frac)√(35)6x) + C_2 sin((frac)√(35)6x) ) + (3)/(8) cos((x) + (1)/(8) sin((x)))
[GEN] Generating candidate 3/3 ...
[CAND] y(x) = C_1 e^frac-1 + √(37)6 x + C_2 e^frac-1 - √(37)6 x + (3)/(8) cos((x) + (1)/(8) sin((x)))
[RESULT] 🏁 Majority: y = C_1 e^-x + C_2 x e^-x + (1)/(4) e^x
[REFINED] ✅ Refined: y = (C_1 + (1)/(4)) e^x + (C_2 - 1) e^-x + x
[FINAL] 📦 y = (C_1 + (1)/(4)) e^x + (C_2 - 1) e^-x + x


RSA-solving:  25%|█████████████████▌                                                    | 1/4 [08:45<26:17, 525.97s/it]

[17:32:57] 
*******************************************************

[🧩] Start solving ODE → y'=-1993+1078x

*******************************************************

[GEN] Generating candidate 1/3 ...
[CAND] y(x) = 539x^2 - 1993x + C
[GEN] Generating candidate 2/3 ...
[CAND] y = -1993x + 539x^2 + C
[GEN] Generating candidate 3/3 ...


RSA-solving:  25%|█████████████████▌                                                    | 1/4 [09:37<28:51, 577.14s/it]
